In [5]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error, r2_score

#day=dataset1
water = pd.read_csv(r"C:\Users\anush\Downloads\water_potability.csv")
water = water.fillna(water.median())         
x= water.drop("Potability", axis=1)
y = water["Potability"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

water_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
water_model.fit(x_train, y_train)
y_pred = water_model.predict(x_test)

print("WATER")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

joblib.dump(water_model, "water_model.pkl")

#dataset2
heart = pd.read_csv(r"C:\Users\anush\Downloads\heart.csv")

x1= heart.drop(["HeartDisease", "Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"], axis=1)
y1 = heart["HeartDisease"]

x1_train, x1_test, y1_train, y1_test = train_test_split(x1, y1, test_size=0.2, random_state=42)

heart_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
heart_model.fit(x1_train, y1_train)
y1_pred = heart_model.predict(x1_test)

print("HEART ")
print("Accuracy:", accuracy_score(y1_test, y1_pred))
print("MSE     :", mean_squared_error(y1_test, y1_pred))
print("MAE     :", mean_absolute_error(y1_test, y1_pred))
print("R2      :", r2_score(y1_test, y1_pred))

joblib.dump(heart_model, "heart_model.pkl")

WATER
Accuracy: 0.6280487804878049
MSE: 0.3719512195121951
MAE: 0.3719512195121951
R2: -0.5922330097087378
HEART 
Accuracy: 0.7391304347826086
MSE     : 0.2608695652173913
MAE     : 0.2608695652173913
R2      : -0.07197475421774469


['heart_model.pkl']

In [7]:
import gradio as gr
import joblib

water_model = joblib.load("water_model.pkl")
heart_model = joblib.load("heart_model.pkl")

water_ui = gr.Interface(
    fn=lambda *v: "Water is SAFE to drink" if water_model.predict([list(v)])[0] == 1 else "Water is NOT safe to drink",
    inputs=[gr.Number(label=n) for n in ["ph", "Hardness", "Solids", "Chloramines", "Sulfate",
                                         "Conductivity", "Organic_carbon", "Trihalomethanes", "Turbidity"]],
    outputs="text",
)

heart_ui = gr.Interface(
    fn=lambda *v: "Heart disease is likely" if heart_model.predict([list(v)])[0] == 1 else "Heart disease is unlikely",
    inputs=[gr.Number(label=n) for n in ["Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak"]],
    outputs="text",
)

app = gr.TabbedInterface([water_ui, heart_ui], ["Water Quality", "Heart Disease"])
app.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
from flask import Flask, request, render_template_string
import joblib

app = Flask(__name__)
water_model = joblib.load("water_model.pkl")
heart_model = joblib.load("heart_model.pkl")

water_features = ["ph", "Hardness", "Solids", "Chloramines", "Sulfate",
                  "Conductivity", "Organic_carbon", "Trihalomethanes", "Turbidity"]
heart_features = ["Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak"]

PAGE = """
<h1>Logistic Regression App</h1>
<a href="/water">Water Quality</a> | <a href="/heart">Heart Disease</a>
{% if features %}
  <h2>{{ title }}</h2>
  <form method="post">
    {% for f in features %}
      {{ f }}: <input type="number" step="any" name="{{ f }}" required><br><br>
    {% endfor %}
    <button type="submit">Predict</button>
  </form>
  {% if result %}<h2>Result: {{ result }}</h2>{% endif %}
{% endif %}
"""

@app.route("/")
def home():
    return render_template_string(PAGE)

@app.route("/water", methods=["GET", "POST"])
def water():
    result = None
    if request.method == "POST":
        row = [[float(request.form[f]) for f in water_features]]
        result = "Water is SAFE to drink" if water_model.predict(row)[0] == 1 else "Water is NOT safe to drink"
    return render_template_string(PAGE, title="Water Quality", features=water_features, result=result)

@app.route("/heart", methods=["GET", "POST"])
def heart():
    result = None
    if request.method == "POST":
        row = [[float(request.form[f]) for f in heart_features]]
        result = "Heart disease is likely" if heart_model.predict(row)[0] == 1 else "Heart disease is unlikely"
    return render_template_string(PAGE, title="Heart Disease", features=heart_features, result=result)

app.run(debug=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [13/Oct/2026 22:34:06] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Oct/2026 22:34:07] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Oct/2026 22:34:08] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [13/Oct/2026 22:34:10] "GET /water HTTP/1.1" 200 -
C:\Users\anush\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\anush\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
127.0.0.1 - - [13/Oct/2026 22:41:06] "GET /heart HTTP/1.1" 200 -
C:\Users\anush\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
127.0.0.1 - - [13/Oct/2026 22:41:32] "POST /heart HTTP/1.1" 200 -
